# 🌊 FloodWatch AI — One-File Training Notebook

## Before you run:
1. **Runtime → Change runtime type → T4 GPU**
2. Upload your images folder using the 📁 Files panel on the left:
   - Click the folder icon → Upload → select all your images
   - They will land in `/content/images/`
3. Fill in your **Gemini API key** in Cell 1
4. Run All (Runtime → Run all)

## What happens automatically:
| Cell | What it does |
|------|-------------|
| 1 | Config — fill your Gemini key here |
| 2 | Install packages |
| 3 | Auto-label all images with Gemini Vision |
| 4 | Split: first 400 = train, rest = test |
| 5 | Download your existing model from GitHub |
| 6 | Fine-tune the model on 400 images |
| 7 | Test accuracy on held-out images |
| 8 | ⭐ OPTIONAL — Deep metrics (ROC, GradCAM, per-class, error analysis) |
| 9 | **Download updated model + all reports** to your PC |


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — FILL IN YOUR SETTINGS HERE
# ═══════════════════════════════════════════════════════════════════

# ── REQUIRED ────────────────────────────────────────────────────────
GEMINI_API_KEY  = ''   # get free key at https://aistudio.google.com/apikey

# ── OPTIONAL — leave as-is if unsure ────────────────────────────────
IMAGES_DIR      = '/content/images'    # where you uploaded your images
TRAIN_COUNT     = 400                  # first N images → training
                                        # remaining images → test
GEMINI_DELAY    = 4.0                  # seconds between Gemini calls (free tier = 15/min)
FINE_TUNE_LR    = 0.0001               # low LR = fine-tune, not retrain from scratch
FINE_TUNE_EPOCHS= 15
BATCH_SIZE      = 16
IMAGE_SIZE      = 224

# ── Verify GPU ────────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
if not GEMINI_API_KEY:
    print('\n⚠️  GEMINI_API_KEY is empty — labelling will be skipped!')
    print('   Get a free key at https://aistudio.google.com/apikey')
else:
    print('✅ Gemini key set')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Install packages
# ═══════════════════════════════════════════════════════════════════
!pip install -q google-generativeai pillow tqdm
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
print('✅ All packages ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Auto-label ALL images with Gemini Vision
#           Output: /content/labels.csv
# ═══════════════════════════════════════════════════════════════════
import csv, json, os, re, time
from pathlib import Path
from PIL     import Image

IMAGES_PATH = Path(IMAGES_DIR)
all_images  = sorted(
    list(IMAGES_PATH.glob('*.jpg'))  +
    list(IMAGES_PATH.glob('*.jpeg')) +
    list(IMAGES_PATH.glob('*.png'))
)

print(f'Found {len(all_images)} images in {IMAGES_PATH}')
assert len(all_images) >= 10, 'Upload at least 10 images'

LABELS_CSV  = Path('/content/labels.csv')
RISK_ORDER  = {'NO FLOOD':0,'LOW RISK':1,'MODERATE':2,'HIGH RISK':3,'CRITICAL':4}

PROMPT = """You are a flood assessment AI for Bengaluru municipality.
Look at this road image and return ONLY valid JSON (no markdown, no explanation):
{
  \"flood_detected\": true or false,
  \"water_depth_cm\": 0-500 (0 if no flood),
  \"risk_level\": \"NO FLOOD\" or \"LOW RISK\" or \"MODERATE\" or \"HIGH RISK\" or \"CRITICAL\",
  \"water_coverage_pct\": 0.0-1.0,
  \"confidence\": 0.0-1.0
}
Rules: NO FLOOD=0cm, LOW=1-15cm, MODERATE=15-35cm, HIGH=35-60cm, CRITICAL>60cm.
If uncertain, pick HIGHER risk. Return ONLY the JSON object."""

# ── Label with Gemini ────────────────────────────────────────────
already_done = set()
if LABELS_CSV.exists():
    already_done = {r['filename'] for r in csv.DictReader(open(LABELS_CSV))}
    print(f'Resuming — {len(already_done)} already labelled')

out_f  = open(LABELS_CSV, 'a' if already_done else 'w', newline='')
writer = csv.DictWriter(out_f, fieldnames=[
    'filename','flood_detected','water_depth_cm',
    'risk_level','water_coverage_pct','confidence'
])
if not already_done:
    writer.writeheader()

errors = []

if GEMINI_API_KEY:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    gemini = genai.GenerativeModel('gemini-1.5-flash')

    for i, img_path in enumerate(all_images):
        fname = img_path.name
        print(f'[{i+1:03d}/{len(all_images)}] {fname}', end=' ... ', flush=True)

        if fname in already_done:
            print('skipped'); continue

        try:
            img  = Image.open(img_path).convert('RGB')
            resp = gemini.generate_content([PROMPT, img])
            raw  = re.sub(r'```[a-z]*\s*','', resp.text.strip()).strip('`').strip()
            data = json.loads(raw)
            writer.writerow({
                'filename':          fname,
                'flood_detected':    bool(data['flood_detected']),
                'water_depth_cm':    float(data['water_depth_cm']),
                'risk_level':        str(data['risk_level']).upper().strip(),
                'water_coverage_pct':float(data.get('water_coverage_pct', 0)),
                'confidence':        float(data.get('confidence', 0.8)),
            })
            out_f.flush()
            print(f"{data['risk_level']:12s}  {data['water_depth_cm']}cm")
        except json.JSONDecodeError:
            errors.append(fname); print('❌ bad JSON')
        except Exception as e:
            errors.append(fname); print(f'❌ {e}')

        time.sleep(GEMINI_DELAY)

out_f.close()

# Load all labels
all_labels = list(csv.DictReader(open(LABELS_CSV)))
print(f'\n✅ Labelling done: {len(all_labels)} images labelled, {len(errors)} errors')

from collections import Counter
dist = Counter(r['risk_level'] for r in all_labels)
print('\nClass distribution:')
for risk in ['NO FLOOD','LOW RISK','MODERATE','HIGH RISK','CRITICAL']:
    count = dist.get(risk, 0)
    print(f'  {risk:12s}: {count:3d}  {"█"*(count//3)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Split: first 400 → TRAIN, rest → TEST
# ═══════════════════════════════════════════════════════════════════
import random

# Only use rows where the image actually exists
valid_labels = [
    r for r in all_labels
    if (Path(IMAGES_DIR) / r['filename']).exists()
]

# Shuffle with fixed seed so split is reproducible
random.seed(42)
random.shuffle(valid_labels)

train_labels = valid_labels[:TRAIN_COUNT]
test_labels  = valid_labels[TRAIN_COUNT:]

print(f'Total valid images : {len(valid_labels)}')
print(f'Train set          : {len(train_labels)}  (first {TRAIN_COUNT})')
print(f'Test set           : {len(test_labels)}  (remaining)')

assert len(train_labels) >= 10, f'Need at least 10 training images, got {len(train_labels)}'
assert len(test_labels)  >= 4,  f'Need at least 4 test images. Upload more images.'

# Show test set distribution
from collections import Counter
test_dist = Counter(r['risk_level'] for r in test_labels)
print(f'\nTest set breakdown:')
for risk in ['NO FLOOD','LOW RISK','MODERATE','HIGH RISK','CRITICAL']:
    c = test_dist.get(risk, 0)
    if c: print(f'  {risk:12s}: {c}')
print('\n✅ Split ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Download your existing model from GitHub
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys
from pathlib import Path

REPO_DIR   = Path('/content/flood-depth-estimator')
MODEL_PATH = '/content/best_flood_model_water_aware.pth'

# ── Clone repo to get training code ──────────────────────────────
if not REPO_DIR.exists():
    print('Cloning repo...')
    r = subprocess.run(
        ['git', 'clone',
         'https://github.com/Mishra-Kaumod/flood-depth-estimator.git',
         str(REPO_DIR)],
        capture_output=True, text=True
    )
    print(r.stdout or r.stderr)

sys.path.insert(0, str(REPO_DIR))

# ── Get model via Git LFS ─────────────────────────────────────────
# Install LFS and pull the .pth file
!apt-get install -q git-lfs
!git -C {str(REPO_DIR)} lfs pull

lfs_model = REPO_DIR / 'models' / 'best_flood_model_water_aware.pth'

if lfs_model.exists() and lfs_model.stat().st_size > 1_000_000:
    import shutil
    shutil.copy(str(lfs_model), MODEL_PATH)
    size_mb = Path(MODEL_PATH).stat().st_size / 1e6
    print(f'\n✅ Model ready: {MODEL_PATH}  ({size_mb:.1f} MB)')
else:
    print('\n⚠️  LFS pull got a pointer file, not the real model.')
    print('   MANUAL FIX: Upload best_flood_model_water_aware.pth directly to Colab:')
    print('   Files panel (left) → Upload → select the .pth from your PC')
    print('   Then re-run this cell.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Fine-tune the existing model on your 400 training images
# ═══════════════════════════════════════════════════════════════════
import csv, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision       import transforms, models
from pathlib           import Path
from PIL               import Image
from tqdm              import tqdm

RISK_TO_IDX = {'NO FLOOD':0,'LOW RISK':1,'MODERATE':2,'HIGH RISK':3,'CRITICAL':4}
IDX_TO_RISK = {v:k for k,v in RISK_TO_IDX.items()}
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ── Dataset ───────────────────────────────────────────────────────
class FloodDS(Dataset):
    def __init__(self, rows, img_dir, transform):
        self.rows    = rows
        self.img_dir = Path(img_dir)
        self.tf      = transform
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r   = self.rows[i]
        img = Image.open(self.img_dir / r['filename']).convert('RGB')
        lbl = RISK_TO_IDX.get(r['risk_level'].upper().strip(), 0)
        return self.tf(img), torch.tensor(lbl, dtype=torch.long)

train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Use 10% of train set as mini-validation during training
n_mini_val  = max(4, int(len(train_labels)*0.10))
mini_val    = train_labels[-n_mini_val:]
mini_train  = train_labels[:-n_mini_val]

tr_dl  = DataLoader(FloodDS(mini_train, IMAGES_DIR, train_tf), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_dl = DataLoader(FloodDS(mini_val,   IMAGES_DIR, val_tf),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Training on {len(mini_train)} images, mini-val on {len(mini_val)} images')

# ── Build model ───────────────────────────────────────────────────
model = models.efficientnet_b0(weights=None)
in_f  = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, 5))

# Load existing checkpoint
ckpt = torch.load(MODEL_PATH, map_location='cpu')
sd   = ckpt.get('model_state_dict', ckpt)
missing, _ = model.load_state_dict(sd, strict=False)
print(f'Loaded checkpoint  ({len(missing)} keys adjusted)')

# Partial freeze: train top 40% of layers only
params_list = list(model.named_parameters())
freeze_up   = int(len(params_list) * 0.60)
for i, (_, p) in enumerate(params_list):
    p.requires_grad = (i >= freeze_up)
trainable = sum(p.requires_grad for _,p in model.named_parameters())
print(f'Trainable layers   : {trainable}/{len(params_list)}')

model = model.to(DEVICE)

# ── Training ──────────────────────────────────────────────────────
optimizer  = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=FINE_TUNE_LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', factor=0.5, patience=3)
criterion  = nn.CrossEntropyLoss()
scaler     = torch.cuda.amp.GradScaler() if DEVICE=='cuda' else None

best_acc   = 0.0
best_epoch = 0
no_improve = 0
history    = []
UPDATED_MODEL = '/content/flood_model_updated.pth'

print(f'\nFine-tuning for {FINE_TUNE_EPOCHS} epochs  LR={FINE_TUNE_LR}')
print('='*60)

for epoch in range(FINE_TUNE_EPOCHS):
    # Train
    model.train()
    tr_loss = 0
    for imgs, lbls in tqdm(tr_dl, desc=f'Ep{epoch+1:02d} train', leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                loss = criterion(model(imgs), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss = criterion(model(imgs), lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        tr_loss += loss.item()

    # Validate
    model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, lbls in val_dl:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            preds   = model(imgs).argmax(1)
            correct += (preds==lbls).sum().item(); total += len(lbls)
    val_acc = correct / max(total, 1)
    scheduler.step(val_acc)
    history.append({'epoch':epoch+1,'train_loss':round(tr_loss/len(tr_dl),4),'val_acc':round(val_acc,4)})
    print(f'Epoch {epoch+1:02d}/{FINE_TUNE_EPOCHS}  loss={tr_loss/len(tr_dl):.4f}  val_acc={val_acc:.3f}', end='')

    if val_acc > best_acc:
        best_acc = val_acc; best_epoch = epoch+1; no_improve = 0
        torch.save({'model_state_dict': model.state_dict(),
                    'val_acc': val_acc, 'epoch': epoch}, UPDATED_MODEL)
        print('  ✅ saved')
    else:
        no_improve += 1
        print(f'  (no improve {no_improve}/{5})')
        if no_improve >= 5: print('Early stop'); break

print(f'\n🏁 Best val_acc = {best_acc:.3f}  at epoch {best_epoch}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Test accuracy on HELD-OUT test images
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Load best model
best_ckpt = torch.load(UPDATED_MODEL, map_location=DEVICE)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()

# Run on test set
test_dl = DataLoader(
    FloodDS(test_labels, IMAGES_DIR, val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

all_preds = []; all_true = []
with torch.no_grad():
    for imgs, lbls in test_dl:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu().tolist()
        all_preds.extend(preds)
        all_true.extend(lbls.tolist())

# ── Metrics ───────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
!pip install -q scikit-learn
from sklearn.metrics import classification_report, confusion_matrix

classes     = ['NO FLOOD','LOW RISK','MODERATE','HIGH RISK','CRITICAL']
present     = sorted(set(all_true + all_preds))
present_names = [classes[i] for i in present]

overall_acc = sum(p==t for p,t in zip(all_preds,all_true)) / len(all_true)

# Flood detection (binary)
true_flood  = [t > 0 for t in all_true]
pred_flood  = [p > 0 for p in all_preds]
tp = sum(pf and tf for pf,tf in zip(pred_flood,true_flood))
fp = sum(pf and not tf for pf,tf in zip(pred_flood,true_flood))
fn = sum(not pf and tf for pf,tf in zip(pred_flood,true_flood))
prec = tp/(tp+fp) if (tp+fp)>0 else 0
rec  = tp/(tp+fn) if (tp+fn)>0 else 0
f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0

# Depth estimation (from risk tier)
RISK_DEPTH_CM = {0:0, 1:8, 2:25, 3:47, 4:80}   # midpoints per tier
true_depths  = [RISK_DEPTH_CM[t] for t in all_true]
pred_depths  = [RISK_DEPTH_CM[p] for p in all_preds]
mae_cm       = np.mean([abs(p-t) for p,t in zip(pred_depths,true_depths)])

# ── Print report ─────────────────────────────────────────────────
print('\n' + '='*58)
print('  TEST ACCURACY REPORT')
print('='*58)
print(f'  Test images        : {len(test_labels)}')
print(f'  Overall accuracy   : {overall_acc*100:.1f}%   target ≥80%  {"✅" if overall_acc>=0.80 else "❌"}')
print(f'  Flood detection F1 : {f1:.3f}         target ≥0.85 {"✅" if f1>=0.85 else "❌"}')
print(f'  Depth MAE          : {mae_cm:.1f} cm      target ≤8cm  {"✅" if mae_cm<=8 else "❌"}')
print('='*58)
print()
print(classification_report(all_true, all_preds, target_names=present_names, zero_division=0))

# ── Confusion matrix ─────────────────────────────────────────────
cm  = confusion_matrix(all_true, all_preds, labels=present)
fig, ax = plt.subplots(figsize=(7,5))
im  = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(present_names))); ax.set_xticklabels(present_names, rotation=30, ha='right')
ax.set_yticks(range(len(present_names))); ax.set_yticklabels(present_names)
for i in range(len(present)):
    for j in range(len(present)):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j]>cm.max()/2 else 'black', fontsize=12)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix  (accuracy={overall_acc*100:.1f}%)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

# ── Training curves ───────────────────────────────────────────────
ep = [r['epoch']     for r in history]
tl = [r['train_loss']for r in history]
va = [r['val_acc']   for r in history]
fig2, (a1,a2) = plt.subplots(1,2,figsize=(12,4))
a1.plot(ep, tl, 'steelblue'); a1.set_title('Train Loss'); a1.grid(alpha=.3)
a2.plot(ep, va, 'seagreen');  a2.set_title('Val Accuracy')
a2.axhline(0.80, color='gold', ls=':', label='80% target')
a2.axhline(best_acc, color='red', ls='--', label=f'Best {best_acc:.3f}')
a2.legend(); a2.grid(alpha=.3)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

all_pass = (overall_acc >= 0.80 and f1 >= 0.85 and mae_cm <= 8)
if all_pass:
    print('\n🎉 ALL TARGETS MET — model is production-ready!')
else:
    print('\n⚠️  Some targets missed — see recommendations in Cell 8')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — ⭐ OPTIONAL: Deep Metrics Evaluation
#
# SET THIS TO True TO RUN  ↓
RUN_DEEP_METRICS = True
# ═══════════════════════════════════════════════════════════════════
if not RUN_DEEP_METRICS:
    print('Skipped — set RUN_DEEP_METRICS = True to run')
else:
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    import torch, os
    from pathlib import Path
    from PIL     import Image
    from torchvision import transforms
    !pip install -q scikit-learn
    from sklearn.metrics import (
        classification_report, confusion_matrix,
        roc_curve, auc, precision_recall_curve
    )

    os.makedirs('/content/metrics', exist_ok=True)
    MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
    RISK_NAMES = ['NO FLOOD','LOW RISK','MODERATE','HIGH RISK','CRITICAL']
    RISK_CM    = {0:0, 1:8, 2:25, 3:47, 4:80}
    RISK_COLORS= ['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad']

    # ── Collect softmax probabilities for all test images ──────────
    val_tf_local = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

    model.eval()
    probs_all, preds_all, true_all = [], [], []
    filenames_all = []

    with torch.no_grad():
        for row in test_labels:
            img_path = Path(IMAGES_DIR) / row['filename']
            if not img_path.exists(): continue
            img  = Image.open(img_path).convert('RGB')
            inp  = val_tf_local(img).unsqueeze(0).to(DEVICE)
            out  = torch.softmax(model(inp), dim=1).squeeze().cpu().numpy()
            true_idx = RISK_TO_IDX.get(row['risk_level'].upper().strip(), 0)
            probs_all.append(out)
            preds_all.append(int(out.argmax()))
            true_all.append(true_idx)
            filenames_all.append(row['filename'])

    probs_all = np.array(probs_all)   # shape (N, 5)
    print(f'Evaluated {len(true_all)} test images')

    # ══════════════════════════════════════════════════════════════
    # METRIC 1 — Per-class Precision / Recall / F1
    # ══════════════════════════════════════════════════════════════
    present     = sorted(set(true_all + preds_all))
    pnames      = [RISK_NAMES[i] for i in present]
    report_str  = classification_report(
        true_all, preds_all, target_names=pnames,
        labels=present, zero_division=0
    )
    print('\n── Per-class Report ────────────────────────────────────')
    print(report_str)
    with open('/content/metrics/classification_report.txt','w') as f:
        f.write(report_str)

    # ══════════════════════════════════════════════════════════════
    # METRIC 2 — Confidence Distribution per class
    # ══════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(1, len(present), figsize=(4*len(present), 4))
    if len(present)==1: axes=[axes]
    for ax, cls_idx in zip(axes, present):
        confs = probs_all[np.array(true_all)==cls_idx, cls_idx]
        if len(confs)==0:
            ax.set_title(f'{RISK_NAMES[cls_idx]}\n(no samples)'); continue
        ax.hist(confs, bins=10, color=RISK_COLORS[cls_idx], alpha=0.8, edgecolor='white')
        ax.axvline(confs.mean(), color='black', linestyle='--', linewidth=1.5,
                   label=f'mean={confs.mean():.2f}')
        ax.set_title(f'{RISK_NAMES[cls_idx]}\nn={len(confs)}')
        ax.set_xlabel('Model confidence'); ax.set_ylabel('Count')
        ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.suptitle('Confidence Distribution per Class', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('/content/metrics/confidence_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ══════════════════════════════════════════════════════════════
    # METRIC 3 — ROC Curve (One-vs-Rest for each class)
    # ══════════════════════════════════════════════════════════════
    from sklearn.preprocessing import label_binarize
    true_bin = label_binarize(true_all, classes=list(range(5)))

    fig, ax = plt.subplots(figsize=(8, 6))
    for cls_idx in present:
        if true_bin[:, cls_idx].sum() == 0: continue
        fpr, tpr, _ = roc_curve(true_bin[:, cls_idx], probs_all[:, cls_idx])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=RISK_COLORS[cls_idx], linewidth=2,
                label=f'{RISK_NAMES[cls_idx]}  AUC={roc_auc:.3f}')
    ax.plot([0,1],[0,1], 'k--', alpha=0.4, label='Random')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve — One-vs-Rest per Risk Level')
    ax.legend(loc='lower right'); ax.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig('/content/metrics/roc_curves.png', dpi=150)
    plt.show()

    # ══════════════════════════════════════════════════════════════
    # METRIC 4 — Precision-Recall Curve (flood vs no-flood)
    # ══════════════════════════════════════════════════════════════
    true_flood_bin  = (np.array(true_all)  > 0).astype(int)
    flood_score     = 1 - probs_all[:, 0]   # P(any flood) = 1 - P(NO FLOOD)
    pr_prec, pr_rec, pr_thresh = precision_recall_curve(true_flood_bin, flood_score)
    pr_auc = auc(pr_rec, pr_prec)

    fig, ax = plt.subplots(figsize=(7,5))
    ax.plot(pr_rec, pr_prec, color='tomato', linewidth=2, label=f'Flood PR  AUC={pr_auc:.3f}')
    ax.axhline(true_flood_bin.mean(), color='gray', linestyle='--',
               alpha=0.6, label=f'Baseline ({true_flood_bin.mean():.2f})')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve — Flood Detection')
    ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig('/content/metrics/precision_recall.png', dpi=150)
    plt.show()

    # ══════════════════════════════════════════════════════════════
    # METRIC 5 — Depth Estimation Error Distribution
    # ══════════════════════════════════════════════════════════════
    true_depths = np.array([RISK_CM[t] for t in true_all])
    pred_depths = np.array([RISK_CM[p] for p in preds_all])
    errors      = pred_depths - true_depths
    mae_cm      = np.mean(np.abs(errors))
    rmse_cm     = np.sqrt(np.mean(errors**2))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.hist(errors, bins=15, color='steelblue', edgecolor='white', alpha=0.85)
    ax1.axvline(0,  color='black',  linestyle='-',  linewidth=1.5, label='Perfect')
    ax1.axvline(mae_cm, color='tomato', linestyle='--', linewidth=1.5, label=f'+MAE={mae_cm:.1f}cm')
    ax1.axvline(-mae_cm,color='tomato', linestyle='--', linewidth=1.5)
    ax1.set_xlabel('Depth Error (cm)'); ax1.set_ylabel('Count')
    ax1.set_title(f'Depth Error Distribution\nMAE={mae_cm:.1f}cm  RMSE={rmse_cm:.1f}cm')
    ax1.legend(); ax1.grid(alpha=.3)

    ax2.scatter(true_depths, pred_depths, alpha=0.6, c='steelblue', edgecolors='white', s=60)
    mn, mx = min(true_depths.min(), pred_depths.min()), max(true_depths.max(), pred_depths.max())
    ax2.plot([mn,mx],[mn,mx], 'k--', alpha=0.5, label='Perfect prediction')
    ax2.set_xlabel('True Depth (cm)'); ax2.set_ylabel('Predicted Depth (cm)')
    ax2.set_title('True vs Predicted Depth'); ax2.legend(); ax2.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig('/content/metrics/depth_error.png', dpi=150)
    plt.show()

    # ══════════════════════════════════════════════════════════════
    # METRIC 6 — Most confident correct & worst mistakes (image grid)
    # ══════════════════════════════════════════════════════════════
    confidences   = probs_all[range(len(preds_all)), preds_all]
    correct_mask  = np.array(preds_all) == np.array(true_all)
    wrong_mask    = ~correct_mask

    def show_image_grid(indices, title, filename, n=6):
        indices = list(indices)[:n]
        if not indices: print(f'No samples for: {title}'); return
        fig, axes = plt.subplots(1, len(indices), figsize=(3*len(indices), 3.5))
        if len(indices)==1: axes=[axes]
        for ax, idx in zip(axes, indices):
            img_path = Path(IMAGES_DIR) / filenames_all[idx]
            img = Image.open(img_path).resize((224,224))
            ax.imshow(img)
            p, t, c = preds_all[idx], true_all[idx], confidences[idx]
            color   = 'green' if p==t else 'red'
            ax.set_title(f'T:{RISK_NAMES[t][:3]}\nP:{RISK_NAMES[p][:3]} {c*100:.0f}%',
                         fontsize=8, color=color)
            ax.axis('off')
        plt.suptitle(title, fontsize=11)
        plt.tight_layout()
        plt.savefig(f'/content/metrics/{filename}', dpi=120, bbox_inches='tight')
        plt.show()

    # Top correct predictions (highest confidence)
    correct_idx = np.where(correct_mask)[0]
    top_correct = correct_idx[np.argsort(-confidences[correct_idx])][:6]
    show_image_grid(top_correct, '✅ Top Correct Predictions (highest confidence)', 'top_correct.png')

    # Worst mistakes (wrong + high confidence = dangerous)
    wrong_idx   = np.where(wrong_mask)[0]
    top_wrong   = wrong_idx[np.argsort(-confidences[wrong_idx])][:6]
    show_image_grid(top_wrong, '❌ Worst Mistakes (wrong + high confidence)', 'worst_mistakes.png')

    # ══════════════════════════════════════════════════════════════
    # METRIC 7 — Summary scorecard
    # ══════════════════════════════════════════════════════════════
    from sklearn.metrics import balanced_accuracy_score
    bal_acc   = balanced_accuracy_score(true_all, preds_all)

    tp = sum(pf and tf for pf,tf in zip(np.array(preds_all)>0, np.array(true_all)>0))
    fp = sum(pf and not tf for pf,tf in zip(np.array(preds_all)>0, np.array(true_all)>0))
    fn = sum(not pf and tf for pf,tf in zip(np.array(preds_all)>0, np.array(true_all)>0))
    prec_fl = tp/(tp+fp) if (tp+fp)>0 else 0
    rec_fl  = tp/(tp+fn) if (tp+fn)>0 else 0
    f1_fl   = 2*prec_fl*rec_fl/(prec_fl+rec_fl) if (prec_fl+rec_fl)>0 else 0

    scores = {
        'Overall Accuracy':     (overall_acc,   0.80, '%'),
        'Balanced Accuracy':    (bal_acc,        0.75, '%'),
        'Flood Detection F1':   (f1_fl,          0.85, ''),
        'Flood Precision':      (prec_fl,        0.80, ''),
        'Flood Recall':         (rec_fl,         0.80, ''),
        'Flood PR-AUC':         (pr_auc,         0.80, ''),
        'Depth MAE (cm)':       (mae_cm,         8.0,  'cm', True),  # True = lower is better
        'Depth RMSE (cm)':      (rmse_cm,        12.0, 'cm', True),
    }

    print('\n' + '═'*58)
    print('  DEEP METRICS SCORECARD')
    print('═'*58)
    all_pass_deep = True
    for name, vals in scores.items():
        val, target, unit = vals[0], vals[1], vals[2]
        lower_better = len(vals) > 3 and vals[3]
        if unit == '%':
            display = f'{val*100:.1f}%'
            tgt_str = f'{target*100:.0f}%'
        elif unit == 'cm':
            display = f'{val:.1f} cm'
            tgt_str = f'{target:.0f} cm'
        else:
            display = f'{val:.3f}'
            tgt_str = f'{target:.2f}'
        passed = (val <= target) if lower_better else (val >= target)
        icon   = '✅' if passed else '❌'
        if not passed: all_pass_deep = False
        print(f'  {icon} {name:25s}: {display:10s}  (target {"≤" if lower_better else "≥"}{tgt_str})')

    print('═'*58)
    if all_pass_deep:
        print('\n🏆 ALL deep metrics passed — excellent model!')
    else:
        print('\n💡 Tips to improve failing metrics:')
        if overall_acc < 0.80:
            print('   • Add more images per under-represented class')
        if f1_fl < 0.85:
            print('   • Add more non-flood images to reduce false positives')
        if mae_cm > 8:
            print('   • Improve depth labels accuracy in labels.csv')
            print('   • Add images with precise depth measurements')
        if bal_acc < 0.75:
            print('   • Dataset is imbalanced — add more images for rare classes')
    print()
    print('  All metric charts saved to /content/metrics/')
    print('  They will be included in your downloaded ZIP')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — Download updated model + all reports to your PC
# ═══════════════════════════════════════════════════════════════════
import shutil, json, os
from datetime import datetime
from google.colab import files

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
acc_pct   = int(best_acc * 100)

pkg_dir = f'/content/floodwatch_model_{timestamp}_acc{acc_pct}'
os.makedirs(pkg_dir, exist_ok=True)

# 1. Updated model (pipeline-ready filename)
shutil.copy(UPDATED_MODEL, f'{pkg_dir}/best_flood_model_water_aware.pth')

# 2. Labels CSV
shutil.copy('/content/labels.csv', f'{pkg_dir}/labels.csv')

# 3. Basic training charts
for chart in ['/content/training_curves.png', '/content/confusion_matrix.png']:
    if os.path.exists(chart):
        shutil.copy(chart, pkg_dir)

# 4. Deep metrics folder (if Cell 8 was run)
if os.path.exists('/content/metrics'):
    shutil.copytree('/content/metrics', f'{pkg_dir}/deep_metrics')
    print('✅ Deep metrics included in package')

# 5. Summary JSON
summary = {
    'trained_at':        datetime.now().isoformat(),
    'train_images':      len(train_labels),
    'test_images':       len(test_labels),
    'best_epoch':        best_epoch,
    'overall_accuracy':  round(overall_acc, 4),
    'flood_f1':          round(f1, 4),
    'depth_mae_cm':      round(float(mae_cm), 1),
    'all_targets_met':   bool(all_pass),
    'model_file':        'best_flood_model_water_aware.pth',
    'deep_metrics_run':  os.path.exists('/content/metrics'),
}
with open(f'{pkg_dir}/accuracy_report.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Zip
zip_path = f'/content/floodwatch_model_{timestamp}_acc{acc_pct}.zip'
shutil.make_archive(zip_path.replace('.zip',''), 'zip', pkg_dir)
print(f'\n📦 Package contents:')
for fname in sorted(os.listdir(pkg_dir)):
    fpath = f'{pkg_dir}/{fname}'
    if os.path.isfile(fpath):
        print(f'   {fname:45s} {os.path.getsize(fpath)/1e6:.1f} MB')
    else:
        sub_files = os.listdir(fpath)
        print(f'   {fname}/  ({len(sub_files)} charts)')

print('\n⬇️  Downloading now...')
files.download(zip_path)

print('\n' + '='*58)
print('  NEXT STEPS AFTER DOWNLOAD')
print('='*58)
print('1. Unzip the downloaded file')
print('2. Copy model to repo:')
print('   best_flood_model_water_aware.pth → flood-depth-estimator\\models\\')
print('3. Commit to GitHub:')
print('   git lfs track "*.pth"')
print('   git add models\\best_flood_model_water_aware.pth')
print(f'  git commit -m "Model acc={acc_pct}% train={len(train_labels)}imgs"')
print('   git push')
print('4. Restart API:  docker restart floodwatch_api')
print(f'\n  Accuracy={overall_acc*100:.1f}%  F1={f1:.3f}  MAE={mae_cm:.1f}cm')